PRIMER PASO: INSTALACIÓN DE PySpark

In [1]:
!pip install pyspark

SEGUNDO PASO: CREACIÓN DE SESION SPARK

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DIRESA_LaLibertad") \
    .getOrCreate()

print("Spark iniciado correctamente")

Spark iniciado correctamente


TERCER PASO: SUBIR EL CSV

In [3]:
from google.colab import files

uploaded = files.upload()

Saving Niños LA LIBERTAD.csv to Niños LA LIBERTAD.csv


CUARTO PASO: CARGAR CON SPARK DATAFRAME

In [4]:
df = spark.read.csv(
    "Niños LA LIBERTAD.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+---+-----------+--------------+-----------+--------------------+-----------+----------+---------+---------+-------------+----+----+---------------+---------+--------+--------------+-----------+-----------+--------------------+------+---+---+---------+-----------+----------------+----+--------------+----------+------+---------+---------------+------------+-----------+---------+-----------+-----------+
| sw|     Diresa|           Red|   Microred|                EESS|  Dpto_EESS| Prov_EESS|Dist_EESS|Renipress|FechaAtencion|Pais|Sexo|FechaNacimiento|EdadMeses|UbigeoPN|DepartamentoPN|ProvinciaPN| DistritoPN|     CentroPobladoPN|Juntos|SIS|Pin|Qaliwarma|Hemoglobina|FechaHemoglobina|Cred|Suplementacion|Consejeria|Sesion|UbigeoREN|DepartamentoREN|ProvinciaREN|DistritoREN|AlturaREN|        Hbc|  Dx_anemia|
+---+-----------+--------------+-----------+--------------------+-----------+----------+---------+---------+-------------+----+----+---------------+---------+--------+--------------+--------

QUINTO PASO: REGISTRAR EL DATAFRAME PARA USAR SPARKSQL

In [5]:
df.createOrReplaceTempView("ninos_hb")

print("Vista SparkSQL creada correctamente")

Vista SparkSQL creada correctamente


SEXTO PASO: CONSULTA CON SPARK_SQL

In [6]:
resultado = spark.sql("""
SELECT
    COUNT(*) AS total_registros,
    ROUND(AVG(Hemoglobina), 2) AS promedio_hemoglobina
FROM ninos_hb
WHERE Hemoglobina IS NOT NULL
""")

resultado.show()

+---------------+--------------------+
|total_registros|promedio_hemoglobina|
+---------------+--------------------+
|          81052|               12.41|
+---------------+--------------------+



SEPTIMO PASO: AGREGACIÓN POR DISTRITO

In [7]:
spark.sql("""
SELECT
    DistritoPN,
    COUNT(*) AS registros,
    ROUND(AVG(Hemoglobina), 2) AS promedio_hb
FROM ninos_hb
WHERE Hemoglobina IS NOT NULL
GROUP BY DistritoPN
ORDER BY registros DESC
LIMIT 10
""").show()

+-----------------+---------+-----------+
|       DistritoPN|registros|promedio_hb|
+-----------------+---------+-----------+
|                 |     8183|      12.39|
|         TRUJILLO|     6327|      11.93|
|      EL PORVENIR|     5910|      11.87|
|     LA ESPERANZA|     5146|      11.91|
|       HUAMACHUCO|     4570|      13.65|
|        HUANCHACO|     2830|      11.77|
|             VIRU|     2339|       11.6|
|FLORENCIA DE MORA|     2291|      11.88|
|             CHAO|     1862|       11.6|
|           CHEPEN|     1694|      11.91|
+-----------------+---------+-----------+



OCTAVO PASO: MEDICION DEL TIEMPO DE CONSULTA

In [8]:
import time

inicio = time.perf_counter()

resultado_tiempo = spark.sql("""
SELECT
    Diresa,
    DistritoPN,
    COUNT(*) AS registros,
    AVG(Hemoglobina) AS promedio_hb
FROM ninos_hb
WHERE UbigeoPN = '130101'
  AND Hemoglobina IS NOT NULL
GROUP BY Diresa, DistritoPN
""")

resultado_tiempo.collect()

fin = time.perf_counter()

tiempo_spark = fin - inicio

print(f"Tiempo de ejecución Spark: {tiempo_spark:.4f} segundos")

Tiempo de ejecución Spark: 1.3455 segundos


TIEMPO APROXIMADO DE EJECUCIÓN: 1.3455 SEGUNDOS

NOVENO PASO: COMPARACIÓN DENTRO

In [9]:
comparacion = {
    "SQL Server (ms)": 24,
    "Spark (ms)": 1345.5
}

print("COMPARACIÓN DE RENDIMIENTO")
print(f"SQL Server: {comparacion['SQL Server (ms)']} ms")
print(f"Spark: {comparacion['Spark (ms)']} ms")
print(f"Spark tardó {comparacion['Spark (ms)'] / comparacion['SQL Server (ms)']:.1f} veces más")

COMPARACIÓN DE RENDIMIENTO
SQL Server: 24 ms
Spark: 1345.5 ms
Spark tardó 56.1 veces más


DECIMO PASO: CONCLUSIÓN DE ESCALABILIDAD

In [10]:
print("""
CONCLUSIÓN:
Para el volumen evaluado, SQL Server presentó menor tiempo de respuesta
gracias al procesamiento local y al índice aplicado sobre UbigeoPN.

Spark resulta más conveniente cuando el volumen de datos aumenta
considerablemente y se requiere procesamiento distribuido, paralelización
y escalabilidad horizontal.
""")


CONCLUSIÓN:
Para el volumen evaluado, SQL Server presentó menor tiempo de respuesta
gracias al procesamiento local y al índice aplicado sobre UbigeoPN.

Spark resulta más conveniente cuando el volumen de datos aumenta
considerablemente y se requiere procesamiento distribuido, paralelización
y escalabilidad horizontal.

